In [ ]:

import os
import re
import gc
import random

import numpy as np
import pandas as pd
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)

from arabert.preprocess import ArabertPreprocessor

SEED = 42

PERTURBATION_SEEDS = [
    42,
    43,
    44,
    45,
    46
]

SEVERITIES = [
    0.10,
    0.20,
    0.30
]

DIMENSIONS = [
    "Textual Accuracy",
    "Completeness",
    "Consistency",
    "Validity",
    "Understandability"
]


TRAIN_PATH = "ArSarcasm_training_data.csv"
TEST_PATH = "ArSarcasm_testing_data.csv"

TEXT_COLUMN = "tweet"
LABEL_COLUMN = "dialect"


MAX_LENGTH = 128

LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32

NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01


# Remove exact train-test text overlap.
REMOVE_TRAIN_TEST_OVERLAP = True


OUTPUT_DIR = "./ArSarcasm_ADI_5_dimensions"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


MODELS = {

    "AraBERTv2": {

        "model_name":
            "aubmindlab/bert-base-arabertv2",

        "arabert_preprocessing":
            True
    },

    "CAMeLBERT-Mix": {

        "model_name":
            "CAMeL-Lab/bert-base-arabic-camelbert-mix",

        "arabert_preprocessing":
            False
    }
}


def set_all_seeds(seed):

    set_seed(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_all_seeds(SEED)


print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


train_original = pd.read_csv(TRAIN_PATH)
test_original = pd.read_csv(TEST_PATH)


print("\nOriginal train shape:", train_original.shape)
print("Original test shape :", test_original.shape)

print("\nTrain columns:", train_original.columns.tolist())
print("Test columns :", test_original.columns.tolist())



for name, dataframe in [
    ("train", train_original),
    ("test", test_original)
]:

    required = {
        TEXT_COLUMN,
        LABEL_COLUMN
    }

    missing = required - set(dataframe.columns)

    if missing:
        raise ValueError(
            f"{name} missing required columns: {missing}"
        )


train_df = (
    train_original[
        [TEXT_COLUMN, LABEL_COLUMN]
    ]
    .copy()
    .rename(
        columns={
            TEXT_COLUMN: "text",
            LABEL_COLUMN: "labels"
        }
    )
)


test_df = (
    test_original[
        [TEXT_COLUMN, LABEL_COLUMN]
    ]
    .copy()
    .rename(
        columns={
            TEXT_COLUMN: "text",
            LABEL_COLUMN: "labels"
        }
    )
)


def clean_dataframe(df):

    df = (
        df
        .dropna(
            subset=[
                "text",
                "labels"
            ]
        )
        .reset_index(drop=True)
    )

    df["text"] = (
        df["text"]
        .astype(str)
        .str.strip()
    )

    df["labels"] = (
        df["labels"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df = (
        df[
            df["text"] != ""
        ]
        .reset_index(drop=True)
    )

    return df


train_df = clean_dataframe(train_df)
test_df = clean_dataframe(test_df)




test_text_set = set(
    test_df["text"]
)


overlap_mask = (
    train_df["text"]
    .isin(test_text_set)
)


n_overlap = int(
    overlap_mask.sum()
)


print(
    "\nExact train-test overlapping tweets:",
    n_overlap
)


if REMOVE_TRAIN_TEST_OVERLAP and n_overlap > 0:

    train_df = (
        train_df[
            ~overlap_mask
        ]
        .reset_index(drop=True)
    )

    print(
        "Removed overlapping tweets from training."
    )


print(
    "\nFinal train shape:",
    train_df.shape
)

print(
    "Final test shape:",
    test_df.shape
)


print("\nTRAIN DIALECT DISTRIBUTION")

print(
    train_df[
        "labels"
    ]
    .value_counts()
)


print("\nTEST DIALECT DISTRIBUTION")

print(
    test_df[
        "labels"
    ]
    .value_counts()
)


train_classes = set(
    train_df["labels"].unique()
)

test_classes = set(
    test_df["labels"].unique()
)


unknown_test_classes = (
    test_classes
    -
    train_classes
)


if unknown_test_classes:

    raise ValueError(
        "Test contains unseen dialect classes: "
        f"{unknown_test_classes}"
    )


label_encoder = LabelEncoder()


train_df["labels"] = (
    label_encoder
    .fit_transform(
        train_df["labels"]
    )
)


test_df["labels"] = (
    label_encoder
    .transform(
        test_df["labels"]
    )
)


num_labels = len(
    label_encoder.classes_
)


id2label = {

    i: str(label)

    for i, label
    in enumerate(
        label_encoder.classes_
    )
}


label2id = {

    str(label): i

    for i, label
    in enumerate(
        label_encoder.classes_
    )
}


print(
    "\nNumber of dialect classes:",
    num_labels
)


print("\nLabel mapping:")

for idx, label in id2label.items():
    print(idx, "->", label)


# Save mapping
pd.DataFrame({

    "label_id":
        range(num_labels),

    "dialect":
        label_encoder.classes_

}).to_csv(

    os.path.join(
        OUTPUT_DIR,
        "dialect_label_mapping.csv"
    ),

    index=False
)



train_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "official_train_processed.csv"
    ),

    index=False
)


test_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "official_test_processed.csv"
    ),

    index=False
)


def stochastic_count(
    target,
    rng
):

    base = int(
        np.floor(target)
    )

    fraction = (
        target - base
    )

    if rng.random() < fraction:
        base += 1

    return base

ARABIC_PATTERN = re.compile(

    r"[\u0621-\u063A"
    r"\u0641-\u064A"
    r"\u0671-\u06D3"
    r"\u06FA-\u06FC]+"
)


def get_arabic_span(
    token,
    min_len=3
):

    if not isinstance(token, str):
        return None

    matches = list(
        ARABIC_PATTERN.finditer(token)
    )

    eligible = [

        match

        for match in matches

        if len(match.group()) >= min_len
    ]

    if not eligible:
        return None

    return max(
        eligible,
        key=lambda x: len(x.group())
    )



ARABIC_CHARS = list(
    "ابتثجحخدذرزسشصضطظعغفقكلمنهوي"
    "أإآؤئءىة"
)


ORTHOGRAPHIC_ALTERNATIVES = {

    "ا": {"أ", "إ", "آ"},
    "أ": {"ا", "إ", "آ"},
    "إ": {"ا", "أ", "آ"},
    "آ": {"ا", "أ", "إ"},

    "ي": {"ى"},
    "ى": {"ي"}
}


def delete_char(word, rng):

    if len(word) < 2:
        return word

    idx = rng.randrange(len(word))

    return (
        word[:idx]
        +
        word[idx + 1:]
    )


def insert_char(word, rng):

    idx = rng.randrange(
        len(word) + 1
    )

    char = rng.choice(
        ARABIC_CHARS
    )

    return (
        word[:idx]
        +
        char
        +
        word[idx:]
    )


def substitute_char(word, rng):

    idx = rng.randrange(
        len(word)
    )

    original = word[idx]

    forbidden = {
        original
    }

    forbidden.update(
        ORTHOGRAPHIC_ALTERNATIVES.get(
            original,
            set()
        )
    )

    candidates = [

        c

        for c in ARABIC_CHARS

        if c not in forbidden
    ]

    if not candidates:
        return word

    replacement = rng.choice(
        candidates
    )

    return (
        word[:idx]
        +
        replacement
        +
        word[idx + 1:]
    )


def transpose_chars(word, rng):

    positions = [

        i

        for i in range(
            len(word) - 1
        )

        if word[i] != word[i + 1]
    ]

    if not positions:
        return word

    idx = rng.choice(
        positions
    )

    chars = list(word)

    chars[idx], chars[idx + 1] = (
        chars[idx + 1],
        chars[idx]
    )

    return "".join(chars)


TYPO_OPERATIONS = [
    delete_char,
    insert_char,
    substitute_char,
    transpose_chars
]


def corrupt_word(word, rng):

    for _ in range(20):

        operation = rng.choice(
            TYPO_OPERATIONS
        )

        result = operation(
            word,
            rng
        )

        if result != word:
            return result

    return insert_char(
        word,
        rng
    )


def perturb_accuracy(
    text,
    severity,
    rng
):

    tokens = text.split()

    eligible = [

        i

        for i, token
        in enumerate(tokens)

        if get_arabic_span(token)
        is not None
    ]

    n = len(eligible)

    if n == 0:
        return text, 0, 0

    k = stochastic_count(
        severity * n,
        rng
    )

    k = min(k, n)

    if k == 0:
        return text, n, 0

    selected = rng.sample(
        eligible,
        k
    )

    changed = 0

    for idx in selected:

        token = tokens[idx]

        match = get_arabic_span(
            token
        )

        word = match.group()

        corrupted = corrupt_word(
            word,
            rng
        )

        tokens[idx] = (
            token[:match.start()]
            +
            corrupted
            +
            token[match.end():]
        )

        if corrupted != word:
            changed += 1

    return (
        " ".join(tokens),
        n,
        changed
    )


def perturb_completeness(
    text,
    severity,
    rng
):

    words = text.split()

    n = len(words)

    if n <= 1:
        return text, n, 0

    k = stochastic_count(
        severity * n,
        rng
    )

    # Keep at least one word.
    k = min(
        k,
        n - 1
    )

    if k == 0:
        return text, n, 0

    selected = set(
        rng.sample(
            range(n),
            k
        )
    )

    output = [

        word

        for idx, word
        in enumerate(words)

        if idx not in selected
    ]

    return (
        " ".join(output),
        n,
        k
    )

CONSISTENCY_MAP = {

    "ا": ["أ", "إ", "آ"],
    "أ": ["ا", "إ", "آ"],
    "إ": ["ا", "أ", "آ"],
    "آ": ["ا", "أ", "إ"],

    "ي": ["ى"],
    "ى": ["ي"]
}


def perturb_consistency(
    text,
    severity,
    rng
):

    chars = list(text)

    eligible = [

        i

        for i, char
        in enumerate(chars)

        if char in CONSISTENCY_MAP
    ]

    n = len(eligible)

    if n == 0:
        return text, 0, 0

    k = stochastic_count(
        severity * n,
        rng
    )

    k = min(k, n)

    if k == 0:
        return text, n, 0

    selected = rng.sample(
        eligible,
        k
    )

    for idx in selected:

        chars[idx] = rng.choice(
            CONSISTENCY_MAP[
                chars[idx]
            ]
        )

    return (
        "".join(chars),
        n,
        k
    )


INVALID_TOKENS = [
    "zxqv999",
    "qvzx777",
    "xqvz555",
    "vzqx333"
]


def perturb_validity(
    text,
    severity,
    rng
):

    words = text.split()

    n = len(words)

    if n == 0:
        return text, 0, 0

    k = stochastic_count(
        severity * n,
        rng
    )

    if k == 0:
        return text, n, 0

    output = words.copy()

    for _ in range(k):

        invalid_token = rng.choice(
            INVALID_TOKENS
        )

        position = rng.randrange(
            len(output) + 1
        )

        output.insert(
            position,
            invalid_token
        )

    return (
        " ".join(output),
        n,
        k
    )



def understandability_count(
    n,
    severity,
    rng
):

    if n < 2:
        return 0

    target = (
        severity * n
    )

    target = min(
        target,
        n
    )

    # Need >=2 words for reordering.
    if target < 2:

        probability = (
            target / 2
        )

        if rng.random() < probability:
            return 2

        return 0

    return min(
        stochastic_count(
            target,
            rng
        ),
        n
    )


def perturb_understandability(
    text,
    severity,
    rng
):

    words = text.split()

    n = len(words)

    if n < 2:
        return text, n, 0

    k = understandability_count(
        n,
        severity,
        rng
    )

    if k < 2:
        return text, n, 0

    selected = None

    # Find positions with at least
    # two distinct words.
    for _ in range(20):

        candidate = sorted(
            rng.sample(
                range(n),
                k
            )
        )

        candidate_words = [
            words[i]
            for i in candidate
        ]

        if len(set(candidate_words)) > 1:

            selected = candidate
            break

    if selected is None:
        return text, n, 0

    selected_words = [
        words[i]
        for i in selected
    ]

    permuted = None

    # Guarantee non-identity permutation.
    for _ in range(30):

        candidate = (
            selected_words.copy()
        )

        rng.shuffle(candidate)

        if candidate != selected_words:

            permuted = candidate
            break

    if permuted is None:
        return text, n, 0

    output = words.copy()

    for idx, word in zip(
        selected,
        permuted
    ):
        output[idx] = word

    perturbed_text = " ".join(
        output
    )

    if perturbed_text == text:
        return text, n, 0

    return (
        perturbed_text,
        n,
        k
    )


PERTURBATION_FUNCTIONS = {

    "Textual Accuracy":
        perturb_accuracy,

    "Completeness":
        perturb_completeness,

    "Consistency":
        perturb_consistency,

    "Validity":
        perturb_validity,

    "Understandability":
        perturb_understandability
}



def create_perturbed_dataframe(
    clean_df,
    dimension,
    severity,
    seed
):

    rng = random.Random(seed)

    function = (
        PERTURBATION_FUNCTIONS[
            dimension
        ]
    )

    perturbed_df = (
        clean_df.copy()
    )

    new_texts = []

    total_eligible = 0
    total_changed = 0
    changed_instances = 0

    for text in clean_df["text"]:

        (
            perturbed_text,
            eligible,
            changed
        ) = function(
            text,
            severity,
            rng
        )

        new_texts.append(
            perturbed_text
        )

        total_eligible += eligible
        total_changed += changed

        if perturbed_text != text:
            changed_instances += 1

    perturbed_df["text"] = (
        new_texts
    )
    assert (
        perturbed_df["labels"]
        .equals(
            clean_df["labels"]
        )
    )

    realized_rate = (
        total_changed
        /
        total_eligible

        if total_eligible > 0

        else 0.0
    )

    changed_instance_rate = (
        changed_instances
        /
        len(clean_df)

        if len(clean_df) > 0

        else 0.0
    )

    diagnostics = {

        "Dimension":
            dimension,

        "Severity":
            severity,

        "Seed":
            seed,

        "Eligible_Units":
            total_eligible,

        "Changed_Units":
            total_changed,

        "Realized_Rate":
            realized_rate,

        "Changed_Instances":
            changed_instances,

        "Changed_Instance_Rate":
            changed_instance_rate
    }

    return (
        perturbed_df,
        diagnostics
    )


PERTURBED_TEST_SETS = {}

ALL_DIAGNOSTICS = []


print(
    "\nGenerating perturbation conditions..."
)


for dimension in DIMENSIONS:

    for severity in SEVERITIES:

        for perturb_seed in PERTURBATION_SEEDS:

            (
                perturbed_df,
                diagnostics
            ) = create_perturbed_dataframe(

                test_df,
                dimension,
                severity,
                perturb_seed
            )

            key = (
                dimension,
                severity,
                perturb_seed
            )

            PERTURBED_TEST_SETS[
                key
            ] = perturbed_df

            ALL_DIAGNOSTICS.append(
                diagnostics
            )


diagnostics_df = pd.DataFrame(
    ALL_DIAGNOSTICS
)


diagnostic_summary = (

    diagnostics_df

    .groupby(
        [
            "Dimension",
            "Severity"
        ],
        as_index=False
    )

    .agg(

        Realized_Rate_Mean=(
            "Realized_Rate",
            "mean"
        ),

        Realized_Rate_SD=(
            "Realized_Rate",
            "std"
        ),

        Changed_Instance_Rate_Mean=(
            "Changed_Instance_Rate",
            "mean"
        )
    )
)


diagnostic_display = (
    diagnostic_summary.copy()
)


diagnostic_display[
    "Severity"
] *= 100

diagnostic_display[
    "Realized_Rate_Mean"
] *= 100

diagnostic_display[
    "Realized_Rate_SD"
] *= 100

diagnostic_display[
    "Changed_Instance_Rate_Mean"
] *= 100


print(
    "\n"
    +
    "=" * 90
)

print(
    "PERTURBATION CHECK BEFORE TRAINING"
)

print(
    "=" * 90
)


display(
    diagnostic_display.round(3)
)

def evaluate_dataset(
    trainer,
    dataset
):

    output = trainer.predict(
        dataset
    )

    predictions = np.argmax(
        output.predictions,
        axis=-1
    )

    labels = (
        output.label_ids
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {

        "accuracy":
            accuracy,

        "macro_f1":
            macro_f1,

        "labels":
            labels,

        "predictions":
            predictions
    }

ALL_RESULTS = []

CLEAN_RESULTS = []



for MODEL_LABEL, CONFIG in MODELS.items():

    print(
        "\n\n"
        +
        "#" * 100
    )

    print(
        "STARTING MODEL:",
        MODEL_LABEL
    )

    print(
        "#" * 100
    )


    set_all_seeds(SEED)


    MODEL_NAME = (
        CONFIG["model_name"]
    )


    if CONFIG[
        "arabert_preprocessing"
    ]:

        print(
            "\nUsing AraBERT preprocessing."
        )

        arabert_preprocessor = (
            ArabertPreprocessor(

                model_name=MODEL_NAME,

                keep_emojis=True
            )
        )


        def model_preprocess(text):

            return (
                arabert_preprocessor
                .preprocess(
                    str(text)
                )
            )

    else:

        print(
            "\nUsing original text for CAMeLBERT."
        )


        def model_preprocess(text):

            return str(text)


    print(
        "\nPreparing clean model inputs..."
    )


    train_model_df = (
        train_df.copy()
    )

    clean_test_model_df = (
        test_df.copy()
    )


    train_model_df["text"] = (
        train_model_df["text"]
        .apply(model_preprocess)
    )

    clean_test_model_df["text"] = (
        clean_test_model_df["text"]
        .apply(model_preprocess)
    )



    tokenizer = (
        AutoTokenizer
        .from_pretrained(
            MODEL_NAME
        )
    )


    model = (
        AutoModelForSequenceClassification
        .from_pretrained(

            MODEL_NAME,

            num_labels=num_labels,

            id2label=id2label,

            label2id=label2id
        )
    )


    def tokenize_function(batch):

        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_LENGTH
        )


    train_dataset = (
        Dataset
        .from_pandas(
            train_model_df,
            preserve_index=False
        )
        .map(
            tokenize_function,
            batched=True
        )
    )


    clean_test_dataset = (
        Dataset
        .from_pandas(
            clean_test_model_df,
            preserve_index=False
        )
        .map(
            tokenize_function,
            batched=True
        )
    )


    collator = (
        DataCollatorWithPadding(
            tokenizer=tokenizer
        )
    )



    training_args = TrainingArguments(

        output_dir=os.path.join(
            OUTPUT_DIR,
            MODEL_LABEL.replace(
                " ",
                "_"
            )
        ),

        learning_rate=LEARNING_RATE,

        per_device_train_batch_size=
            TRAIN_BATCH_SIZE,

        per_device_eval_batch_size=
            EVAL_BATCH_SIZE,

        num_train_epochs=
            NUM_EPOCHS,

        weight_decay=
            WEIGHT_DECAY,

        logging_strategy="epoch",

        save_strategy="no",

        report_to="none",

        seed=SEED,

        data_seed=SEED,

        optim="adamw_torch"
    )


    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=train_dataset,

        data_collator=collator
    )


    print(
        "\n========================================"
    )

    print(
        "TRAINING",
        MODEL_LABEL
    )

    print(
        "========================================"
    )


    trainer.train()



    clean_result = evaluate_dataset(
        trainer,
        clean_test_dataset
    )


    clean_accuracy = (
        clean_result["accuracy"]
    )

    clean_f1 = (
        clean_result["macro_f1"]
    )


    CLEAN_RESULTS.append({

        "Model":
            MODEL_LABEL,

        "Clean_Accuracy":
            clean_accuracy,

        "Clean_Macro_F1":
            clean_f1
    })


    print(
        "\n========================================"
    )

    print(
        MODEL_LABEL,
        "CLEAN RESULTS"
    )

    print(
        "========================================"
    )

    print(
        f"Accuracy : "
        f"{clean_accuracy * 100:.2f}"
    )

    print(
        f"Macro-F1 : "
        f"{clean_f1 * 100:.2f}"
    )



    class_report = classification_report(

        clean_result["labels"],

        clean_result["predictions"],

        labels=list(
            range(num_labels)
        ),

        target_names=[
            id2label[i]
            for i in range(num_labels)
        ],

        output_dict=True,

        zero_division=0
    )


    pd.DataFrame(
        class_report
    ).transpose().to_csv(

        os.path.join(
            OUTPUT_DIR,
            f"{MODEL_LABEL}_clean_class_report.csv"
        )
    )


    def prepare_perturbed_dataset(
        raw_dataframe
    ):

        temp = (
            raw_dataframe.copy()
        )

        temp["text"] = (
            temp["text"]
            .apply(
                model_preprocess
            )
        )

        dataset = (
            Dataset
            .from_pandas(
                temp,
                preserve_index=False
            )
            .map(
                tokenize_function,
                batched=True
            )
        )

        return dataset




    for dimension in DIMENSIONS:

        print(
            "\n\n"
            +
            "=" * 90
        )

        print(
            MODEL_LABEL,
            "-",
            dimension
        )

        print(
            "=" * 90
        )


        for severity in SEVERITIES:

            print(
                "\nSeverity:",
                int(severity * 100),
                "%"
            )


            for perturb_seed in (
                PERTURBATION_SEEDS
            ):

                key = (
                    dimension,
                    severity,
                    perturb_seed
                )

                raw_perturbed_df = (
                    PERTURBED_TEST_SETS[
                        key
                    ]
                )

                perturbed_dataset = (
                    prepare_perturbed_dataset(
                        raw_perturbed_df
                    )
                )

                result = evaluate_dataset(
                    trainer,
                    perturbed_dataset
                )

                perturbed_accuracy = (
                    result["accuracy"]
                )

                perturbed_f1 = (
                    result["macro_f1"]
                )

                delta_f1 = (
                    clean_f1
                    -
                    perturbed_f1
                )


                diagnostic = (
                    diagnostics_df[
                        (
                            diagnostics_df["Dimension"]
                            ==
                            dimension
                        )
                        &
                        (
                            diagnostics_df["Severity"]
                            ==
                            severity
                        )
                        &
                        (
                            diagnostics_df["Seed"]
                            ==
                            perturb_seed
                        )
                    ]
                    .iloc[0]
                )


                ALL_RESULTS.append({

                    "Model":
                        MODEL_LABEL,

                    "Dimension":
                        dimension,

                    "Severity_Percent":
                        int(
                            severity * 100
                        ),

                    "Perturbation_Seed":
                        perturb_seed,

                    "Clean_Accuracy":
                        clean_accuracy,

                    "Clean_Macro_F1":
                        clean_f1,

                    "Perturbed_Accuracy":
                        perturbed_accuracy,

                    "Perturbed_Macro_F1":
                        perturbed_f1,

                    "Delta_F1":
                        delta_f1,

                    "Realized_Rate":
                        diagnostic[
                            "Realized_Rate"
                        ],

                    "Changed_Instance_Rate":
                        diagnostic[
                            "Changed_Instance_Rate"
                        ]
                })


                print(
                    f"Seed {perturb_seed}"
                    f" | F1={perturbed_f1 * 100:.2f}"
                    f" | Delta={delta_f1 * 100:.2f}"
                    f" | Realized={diagnostic['Realized_Rate'] * 100:.2f}%"
                    f" | Changed={diagnostic['Changed_Instance_Rate'] * 100:.2f}%"
                )


                del perturbed_dataset

                gc.collect()



    del trainer
    del model
    del tokenizer
    del train_dataset
    del clean_test_dataset

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


results_df = pd.DataFrame(
    ALL_RESULTS
)


results_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "all_individual_runs.csv"
    ),

    index=False
)


summary_df = (

    results_df

    .groupby(
        [
            "Model",
            "Dimension",
            "Severity_Percent"
        ],

        as_index=False
    )

    .agg(

        Clean_Accuracy=(
            "Clean_Accuracy",
            "first"
        ),

        Clean_Macro_F1=(
            "Clean_Macro_F1",
            "first"
        ),

        Accuracy_Mean=(
            "Perturbed_Accuracy",
            "mean"
        ),

        Accuracy_SD=(
            "Perturbed_Accuracy",
            "std"
        ),

        Macro_F1_Mean=(
            "Perturbed_Macro_F1",
            "mean"
        ),

        Macro_F1_SD=(
            "Perturbed_Macro_F1",
            "std"
        ),

        Delta_F1_Mean=(
            "Delta_F1",
            "mean"
        ),

        Delta_F1_SD=(
            "Delta_F1",
            "std"
        ),

        Realized_Rate_Mean=(
            "Realized_Rate",
            "mean"
        ),

        Changed_Instance_Rate_Mean=(
            "Changed_Instance_Rate",
            "mean"
        ),

        N_Runs=(
            "Perturbation_Seed",
            "count"
        )
    )
)


summary_100 = (
    summary_df.copy()
)


scale_columns = [

    "Clean_Accuracy",
    "Clean_Macro_F1",

    "Accuracy_Mean",
    "Accuracy_SD",

    "Macro_F1_Mean",
    "Macro_F1_SD",

    "Delta_F1_Mean",
    "Delta_F1_SD",

    "Realized_Rate_Mean",
    "Changed_Instance_Rate_Mean"
]


for column in scale_columns:

    summary_100[
        column
    ] *= 100



paper_df = (
    summary_100.copy()
)


paper_df["Macro-F1"] = (

    paper_df[
        "Macro_F1_Mean"
    ].map(
        lambda x:
        f"{x:.2f}"
    )

    +

    " ± "

    +

    paper_df[
        "Macro_F1_SD"
    ].map(
        lambda x:
        f"{x:.2f}"
    )
)


paper_df["Delta-F1"] = (

    paper_df[
        "Delta_F1_Mean"
    ].map(
        lambda x:
        f"{x:.2f}"
    )

    +

    " ± "

    +

    paper_df[
        "Delta_F1_SD"
    ].map(
        lambda x:
        f"{x:.2f}"
    )
)


paper_df[
    "Realized Severity"
] = (

    paper_df[
        "Realized_Rate_Mean"
    ]

    .map(
        lambda x:
        f"{x:.2f}%"
    )
)


paper_table = (

    paper_df[
        [
            "Model",
            "Dimension",
            "Severity_Percent",
            "Clean_Macro_F1",
            "Macro-F1",
            "Delta-F1",
            "Realized Severity"
        ]
    ]

    .sort_values(
        [
            "Model",
            "Dimension",
            "Severity_Percent"
        ]
    )
)


print(
    "\n\n"
    +
    "#" * 110
)

print(
    "FINAL ArSarcasm-v2 ADI RESULTS"
)

print(
    "#" * 110
)


display(
    paper_table
)



dimension_ranking = (

    summary_100

    .groupby(
        [
            "Model",
            "Dimension"
        ],

        as_index=False
    )

    .agg(

        Mean_Delta_F1=(
            "Delta_F1_Mean",
            "mean"
        )
    )

    .sort_values(
        [
            "Model",
            "Mean_Delta_F1"
        ],

        ascending=[
            True,
            False
        ]
    )
)


print(
    "\n"
    +
    "=" * 90
)

print(
    "DATA-QUALITY SENSITIVITY RANKING"
)

print(
    "=" * 90
)


display(
    dimension_ranking.round(3)
)




print(
    "\n"
    +
    "=" * 90
)

print(
    "SEVERITY MONOTONICITY CHECK"
)

print(
    "=" * 90
)


for model_label in MODELS:

    for dimension in DIMENSIONS:

        subset = (

            summary_100[
                (
                    summary_100["Model"]
                    ==
                    model_label
                )
                &
                (
                    summary_100["Dimension"]
                    ==
                    dimension
                )
            ]

            .sort_values(
                "Severity_Percent"
            )
        )


        deltas = (
            subset[
                "Delta_F1_Mean"
            ]
            .values
        )


        monotonic = all(

            deltas[i]
            <=
            deltas[i + 1]

            for i in range(
                len(deltas) - 1
            )
        )


        print(
            f"{model_label:18s}"
            f" | {dimension:20s}"
            f" | {monotonic}"
        )



clean_results_df = pd.DataFrame(
    CLEAN_RESULTS
)


clean_results_df[
    "Clean_Accuracy"
] *= 100

clean_results_df[
    "Clean_Macro_F1"
] *= 100


print(
    "\n"
    +
    "=" * 90
)

print(
    "CLEAN MODEL COMPARISON"
)

print(
    "=" * 90
)


display(
    clean_results_df.round(3)
)


summary_100.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "five_dimensions_summary.csv"
    ),

    index=False
)


paper_table.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "paper_ready_ADI_results.csv"
    ),

    index=False
)


dimension_ranking.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "dimension_sensitivity_ranking.csv"
    ),

    index=False
)


clean_results_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "clean_model_results.csv"
    ),

    index=False
)


diagnostics_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "perturbation_diagnostics.csv"
    ),

    index=False
)


diagnostic_display.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "perturbation_summary.csv"
    ),

    index=False
)


print(
    "\n\nEXPERIMENT COMPLETED."
)

print(
    "\nMain paper file:"
)

print(
    os.path.join(
        OUTPUT_DIR,
        "paper_ready_ADI_results.csv"
    )
)

CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition

Original train shape: (12548, 4)
Original test shape : (3000, 4)

Train columns: ['tweet', 'sarcasm', 'sentiment', 'dialect']
Test columns : ['tweet', 'sarcasm', 'sentiment', 'dialect']

Exact train-test overlapping tweets: 2
Removed overlapping tweets from training.

Final train shape: (12546, 2)
Final test shape: (3000, 2)

TRAIN DIALECT DISTRIBUTION
labels
msa       8561
egypt     2674
gulf       644
levant     624
magreb      43
Name: count, dtype: int64

TEST DIALECT DISTRIBUTION
labels
msa       2323
gulf       322
egypt      306
levant      47
magreb       2
Name: count, dtype: int64

Number of dialect classes: 5

Label mapping:
0 -> egypt
1 -> gulf
2 -> levant
3 -> magreb
4 -> msa

Generating perturbation conditions...

PERTURBATION CHECK BEFORE TRAINING


,Dimension,Severity,Realized_Rate_Mean,Realized_Rate_SD,Changed_Instance_Rate_Mean
0,Completeness,10.0,9.999,0.059,87.413
1,Completeness,20.0,20.000,0.031,95.407
2,Completeness,30.0,30.000,0.028,97.613
3,Consistency,10.0,10.004,0.046,88.847
4,Consistency,20.0,19.994,0.041,95.413
5,Consistency,30.0,30.013,0.035,97.347
6,Textual Accuracy,10.0,9.985,0.058,82.353
7,Textual Accuracy,20.0,19.999,0.032,92.340
8,Textual Accuracy,30.0,30.012,0.049,95.747
9,Understandability,10.0,9.994,0.087,71.093


[2026-09-06 23:22:51,776 - farasapy_logger - WARNING]: Failed to setup cache directory: [Errno 2] No such file or directory: '/root/.cache/farasapy'. Disabling cache.
[2026-09-06 23:22:51,845 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.




####################################################################################################
STARTING MODEL: AraBERTv2
####################################################################################################

Using AraBERT preprocessing.



Preparing clean model inputs...


...


DATA-QUALITY SENSITIVITY RANKING


,Model,Dimension,Mean_Delta_F1
0,AraBERTv2,Completeness,0.299
1,AraBERTv2,Consistency,-0.325
3,AraBERTv2,Understandability,-0.669
2,AraBERTv2,Textual Accuracy,-0.683
4,AraBERTv2,Validity,-2.074
6,CAMeLBERT-Mix,Consistency,0.583
5,CAMeLBERT-Mix,Completeness,0.391
7,CAMeLBERT-Mix,Textual Accuracy,-0.071
8,CAMeLBERT-Mix,Understandability,-0.238
9,CAMeLBERT-Mix,Validity,-0.691



SEVERITY MONOTONICITY CHECK
AraBERTv2          | Textual Accuracy     | False
AraBERTv2          | Completeness         | True
AraBERTv2          | Consistency          | True
AraBERTv2          | Validity             | False
AraBERTv2          | Understandability    | False
CAMeLBERT-Mix      | Textual Accuracy     | False
CAMeLBERT-Mix      | Completeness         | True
CAMeLBERT-Mix      | Consistency          | True
CAMeLBERT-Mix      | Validity             | False
CAMeLBERT-Mix      | Understandability    | False

CLEAN MODEL COMPARISON


,Model,Clean_Accuracy,Clean_Macro_F1
0,AraBERTv2,71.9,36.595
1,CAMeLBERT-Mix,72.8,35.235




EXPERIMENT COMPLETED.

Main paper file:
./ArSarcasm_ADI_5_dimensions/paper_ready_ADI_results.csv
